### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ecoli_proteins",
    dataset_year="1996",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5388M",
    download_description="""

wget https://archive.ics.uci.edu/static/public/39/ecoli.zip && unzip ecoli.zip && rm ecoli.zip ecoli.names
mkdir -p local-data-warehouse/ecoli_proteins && mv ecoli.data local-data-warehouse/ecoli_proteins/
""",
    # References
    academic_reference_bibtex="""@inproceedings{horton1996probabilistic,
  title={A probabilistic classification system for predicting the cellular localization sites of proteins.},
  author={Horton, Paul and Nakai, Kenta},
  booktitle={Ismb},
  volume={4},
  pages={109--115},
  year={1996},
  organization={St. Louis, Missouri, USA}
}
""",
    academic_reference_bibtex_key="horton1996probabilistic",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the dataset from UCI.

- We drop the sequence name column, which is a unique identifier.
- We drop three classes with less than 10 samples each: omL (5), imL (2), and imS (3).
- We drop "chg", which becomes constant after our preprocessing steps.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="class",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

feature_names = [
    "Sequence Name",
    "mcg",
    "gvh",
    "lip",
    "chg",
    "aac",
    "alm1",
    "alm2",
    "class",
]
df = pd.read_csv(dataset_mold.path / "ecoli.data", header=None, names=feature_names, sep="\s+")
print("Loaded data shape:", df.shape)

df = df.drop(columns=[
    "Sequence Name",  # avoid leakage from look the name and otherwise unique identifier.
    "chg", # constant after preprocessing
])

# Drop samples with classes with too little data (less than 10 samples)
df = df[~df["class"].isin(["omL", "imL", "imS"])]

as_cat_type = ["lip", "class"]
df[as_cat_type] = df[as_cat_type].astype("category")


df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()